## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-28-10-34-58 +0000,bbc,Two staff killed in Israeli strike on Gaza med...,https://www.bbc.co.uk/news/articles/cm5y10yydl...
1,2026-11-28-10-30-00 +0000,wsj,The Legend of New York City’s Most Prolific Wa...,https://www.wsj.com/lifestyle/new-york-city-wa...
2,2026-11-28-10-19-00 +0000,wsj,S&P Revises Outlook on France’s Credit Rating ...,https://www.wsj.com/world/europe/s-p-revises-o...
3,2026-11-28-10-14-37 +0000,bbc,Seven students killed and nearly 100 injured i...,https://www.bbc.co.uk/news/articles/ckp80jyz6d...
4,2026-11-28-10-02-17 +0000,nyt,Millions of Children Are Losing Food Stamps,https://www.nytimes.com/2026/11/28/us/politics...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2500/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
10,new,20
197,trump,20
140,thanksgiving,13
350,friday,10
410,court,9


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
135,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...,62
244,2026-11-27-12-24-03 +0000,chitri,Ukraine president urges US to pressure Vladimi...,https://www.chicagotribune.com/2026/11/27/ukra...,53
143,2026-11-27-20-00-00 +0000,missionlocal,"Tenderloin Buzz: Three new businesses, just in...",https://missionlocal.org/2026/11/tenderloin-bu...,52
188,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...,48
106,2026-11-27-21-55-43 +0000,latimes,"Black Friday shoppers line up early, but think...",https://www.latimes.com/business/story/2026-11...,47


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
135,62,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...
145,47,2026-11-27-19-55-08 +0000,chitri,Black Friday brings shoppers out in force in A...,https://www.chicagotribune.com/2026/11/27/blac...
192,46,2026-11-27-16-53-46 +0000,bbc,Canada's top court largely upholds Quebec ban ...,https://www.bbc.co.uk/news/articles/cq3dgpy9le...
188,45,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...
40,34,2026-11-28-05-01-10 +0000,nyt,"Abroad, Spain’s Leader Is a Left-Wing Hero. Wi...",https://www.nytimes.com/2026/11/28/world/europ...
177,34,2026-11-27-17-59-51 +0000,nypost,Iranian woman brutally tortured while on death...,https://nypost.com/2026/11/27/world-news/irani...
118,33,2026-11-27-21-21-19 +0000,nypost,NYC grocery prices jump nearly one-third since...,https://nypost.com/2026/11/27/business/nyc-gro...
88,29,2026-11-27-23-14-37 +0000,nypost,DHS accuses California of endangering ICE offi...,https://nypost.com/2026/11/27/us-news/dhs-accu...
65,27,2026-11-28-01-31-28 +0000,bbc,Two American Airlines planes clip wings before...,https://www.bbc.co.uk/news/articles/cqkgr4g5nr...
181,27,2026-11-27-17-41-21 +0000,cbc,Zelenskyy criticizes Trump for cozying up to P...,https://www.cbc.ca/news/world/zelensky-eu-ukra...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
